In [1]:
# langchain提供的checkpointer的默认实现，基于内存存储
from langgraph.checkpoint.memory import InMemorySaver

from langchain.agents import create_agent

# 创建智能体时指定checkpointer，LangChain会自动帮我们管理历史会话记忆
agent = create_agent(
    "deepseek-chat",
    checkpointer=InMemorySaver()
)

In [12]:
from langchain_core.messages import HumanMessage

#设定thread_id，作为会话标识
config = {"configurable": {"thread_id":"thread_1"}}

#第一次调用，告知AI信息
response = agent.invoke({"messages":[HumanMessage("你好,我叫虎哥，我最喜欢狗狗")]},
                        config #调用时添加thread_id
                        )
print(response['messages'])

[HumanMessage(content='你好,我叫虎哥，我最喜欢狗狗', additional_kwargs={}, response_metadata={}, id='104c32a5-9d96-4c73-b012-4537fe9585ce'), AIMessage(content='你好啊虎哥！很高兴认识你～喜欢狗狗的人通常都特别有爱心，狗狗忠诚又可爱，确实值得喜欢。你平时有养狗吗？还是更喜欢和狗狗玩耍？如果有机会，可以分享一下你和狗狗的故事吗？🐶', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 13, 'total_tokens': 65, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 13}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'e72d3c7f663d3e1383b00b3884cb4501', 'id': '51dc79ff-5678-42a5-9190-c8e038924c5d', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a16cfc-da69-7461-8300-8fdca24fae7f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 52, 'total_tokens': 65, 'input_token

In [ ]:
#第二次调用
response = agent.invoke({"messages":[HumanMessage("我最喜欢什么动物")]},
                        config #调用时添加thread_id
                        )
print(response['messages'])

[HumanMessage(content='你好,我叫虎哥，我最喜欢狗狗', additional_kwargs={}, response_metadata={}, id='104c32a5-9d96-4c73-b012-4537fe9585ce'), AIMessage(content='你好啊虎哥！很高兴认识你～喜欢狗狗的人通常都特别有爱心，狗狗忠诚又可爱，确实值得喜欢。你平时有养狗吗？还是更喜欢和狗狗玩耍？如果有机会，可以分享一下你和狗狗的故事吗？🐶', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 13, 'total_tokens': 65, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 13}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'e72d3c7f663d3e1383b00b3884cb4501', 'id': '51dc79ff-5678-42a5-9190-c8e038924c5d', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a16cfc-da69-7461-8300-8fdca24fae7f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 52, 'total_tokens': 65, 'input_token

In [ ]:
# LangChain也提供了很多持久化存储的checkpointer，例如：
# - SqlLiteSaver ：基于sqlite存储
# - PostgresSaver ：基于Postgres存储
# - CosmosDBSaver ：使用Azure Cosmos DB的实现

import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

# 初始化checkpointer
checkpointer = SqliteSaver(sqlite3.connect("checkpoint.db", check_same_thread=False))
# 自动建表
checkpointer.setup()

# 创建agent
agent = create_agent(
    "deepseek-chat",
    checkpointer=checkpointer,
)

In [ ]:
#langchain中的记忆管理策略：删除 修剪 总结 自定义

#总结记忆
from langchain.agents.middleware import SummarizationMiddleware
from langchain.agents import create_agent
from langchain.agents.middleware import ShellToolMiddleware
from langchain_core.runnables import RunnableConfig

#初始化checkpointer
checkpointer= InMemorySaver()

#初始化中间件
middleware =  SummarizationMiddleware(
            model = "deepseek-chat",
            trigger = ("messages", 3), #  触发时机，当消息数超过3时，进行总结
            keep=("messages", 1)  #  保留的会话数，超过2条
)

#创建智能体
agent = create_agent(
    "deepseek-chat",
    checkpointer=checkpointer, #记忆存储方式
    middleware= [middleware]
)

In [15]:
#调用agent
config = {"configurable":{"thread_id":"thread_2"}}

#制造长会话历史
agent.invoke({"messages":[HumanMessage("你好我是虎哥")]},config=config)
agent.invoke({"messages": "我最喜欢的运动是乒乓"}, config)
agent.invoke({"messages": "我最喜欢的动物是猫猫"}, config)
# 测试效果
final_response = agent.invoke({"messages": "你还记得我吗？"}, config)


for message in final_response["messages"]:
    message.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT
用户（自称"虎哥"）正在进行初次问候与个人信息分享，尚未提出具体任务或目标。目的是建立联系并了解用户需求。目前用户已分享两项个人喜好：最喜欢的运动是乒乓球，最喜欢的动物是猫猫。

## SUMMARY
- 用户自我介绍为"虎哥"。
- 已分享的个人喜好：
  - 最喜欢的运动：乒乓球
  - 最喜欢的动物：猫猫
- AI助手（DeepSeek）已回应问候并介绍自身为AI身份。
- 针对乒乓球话题，AI询问了是打着玩还是有固定球友/参赛，想聊技术、装备还是赛事（用户未回应此问题）。
- 针对猫猫话题，AI询问用户是养猫还是单纯喜欢，想聊养猫经验、品种还是云吸猫（用户未回应此问题）。
- 用户尚未提出任何具体请求、任务或问题，仍未明确核心需求。
- 无关键决策、结论或策略产生；无被拒绝的选项。

## ARTIFACTS
None

## NEXT STEPS
- 等待用户（虎哥）明确其具体需求或任务（可能是乒乓球、猫猫相关话题，或其他请求）。
- 根据用户后续请求提供相应帮助。
================================ Human Message =================================

你还记得我吗？
================================== Ai Message ==================================

虎哥好！当然记得你～ 乒乓球和猫猫，这两样我可都记着呢。

不过上次我问你的问题你还没回我呢：乒乓球是打着玩还是有固定球友？猫猫是自己养还是云吸猫？

今天想聊点啥？还是有别的事儿要我帮忙？
